# Model Conversion: PyTorch ↔ SNEPPX

Convert a small PyTorch model to SNEPPX weights, save checkpoints,
load with `from_pretrained`, and export to ONNX-safe format.

In [ ]:
import numpy as np
import SneppX_ALG as S
HAS_C = S._HAS_C_BACKEND
print('C backend:', HAS_C)

from SneppX_ALG import Transformer
from SneppX_ALG.interface_bindings.conversion import (
    convert_torch_state_dict, convert_sneppx_to_torch,
    export_checkpoint, load_checkpoint, CheckpointFormat,
)

## 1. Load a PyTorch state dict (mocked here)

In [ ]:
# Simulate a tiny torch state dict for a Transformer with dim=32
rng = np.random.default_rng(0)
torch_sd = {
    'embedding.weight': rng.standard_normal((100, 32)).astype(np.float32) * 0.02,
    'layers.0.attention.q_proj.weight': rng.standard_normal((32, 32)).astype(np.float32) * 0.02,
    'layers.0.attention.k_proj.weight': rng.standard_normal((32, 32)).astype(np.float32) * 0.02,
    'layers.0.attention.v_proj.weight': rng.standard_normal((32, 32)).astype(np.float32) * 0.02,
    'layers.0.attention.out_proj.weight': rng.standard_normal((32, 32)).astype(np.float32) * 0.02,
    'layers.0.ffn.up.weight': rng.standard_normal((64, 32)).astype(np.float32) * 0.02,
    'layers.0.ffn.down.weight': rng.standard_normal((32, 64)).astype(np.float32) * 0.02,
    'lm_head.weight': rng.standard_normal((100, 32)).astype(np.float32) * 0.02,
}

model = Transformer(vocab_size=100, dim=32, num_heads=4, num_layers=1,
                    ffn_dim=64, max_seq_len=32)
sneppx_sd = convert_torch_state_dict(torch_sd, model)
for n, p in model.named_parameters():
    if n in sneppx_sd:
        p.data = sneppx_sd[n]
print('converted', len(sneppx_sd), 'tensors')

## 2. Save & reload checkpoint

In [ ]:
ckpt_path = '/tmp/converted_model.snp'
export_checkpoint(model, ckpt_path, fmt=CheckpointFormat.SNEPPX_NATIVE)
print('saved', ckpt_path)

# Reload
model2 = Transformer(vocab_size=100, dim=32, num_heads=4, num_layers=1,
                     ffn_dim=64, max_seq_len=32)
load_checkpoint(model2, ckpt_path)
print('reloaded checkpoint')

## 3. Round-trip torch

In [ ]:
back_to_torch = convert_sneppx_to_torch(model2)
max_diff = max(np.abs(back_to_torch[k] - torch_sd[k]).max()
               for k in torch_sd if k in back_to_torch)
print('max round-trip diff:', round(float(max_diff), 7))

## 4. ONNX export (if `onnx` is installed)

In [ ]:
import importlib.util
if importlib.util.find_spec('onnx'):
    from SneppX_ALG.interface_bindings.conversion import export_onnx
    export_onnx(model2, '/tmp/model.onnx')
    print('exported /tmp/model.onnx')
else:
    print('onnx not installed - skip export')

# Also write a HF config stub for transformers compatibility
from SneppX_ALG.interface_bindings import save_hf_config
save_hf_config('/tmp/hf_config.json',
               vocab_size=100, dim=32, num_heads=4, num_layers=1)